# Reforço técnico — DevOps, CI/CD e GitFlow

Este notebook cobre os fundamentos das semanas 1 a 7 e executa um laboratório distribuído com Git real. A topologia contém um repositório remoto bare e dois clones independentes, representando dois desenvolvedores.

**Não exige internet ou conta no GitHub.** Requer apenas Git e Python instalados.


## Resultados de aprendizagem

Ao final, o estudante deverá conseguir:

- explicar fluxo de valor, CALMS e métricas DORA;
- descrever objetos, referências e estados internos do Git;
- diferenciar clone, fetch, pull e push;
- aplicar os contratos de feature, release e hotfix;
- explicar CI, Continuous Delivery e Continuous Deployment;
- relacionar testes automatizados e Quality Gates;
- demonstrar sincronização, divergência, rejeição non-fast-forward e conflito entre dois clones.


## 1. DevOps como sistema de entrega

DevOps trata o fluxo completo entre alteração de código e operação do serviço. O objetivo técnico é reduzir lead time sem aumentar a taxa de falhas.

| Problema | Consequência | Prática |
|---|---|---|
| Branches longas | Integração tardia | CI e mudanças pequenas |
| Deploy manual | Variação entre ambientes | Pipeline versionado |
| Testes tardios | Retrabalho | Shift-left |
| Falta de telemetria | Recuperação lenta | Logs, métricas e alertas |


### CALMS

- **Culture:** responsabilidade compartilhada e revisão por pares.
- **Automation:** execução repetível de build, testes e deploy.
- **Lean:** lotes pequenos, limite de WIP e redução de filas.
- **Measurement:** métricas de fluxo e confiabilidade.
- **Sharing:** padrões, documentação e aprendizado acessíveis.

### Métricas DORA

| Métrica | Definição operacional | Direção |
|---|---|---|
| Deployment Frequency | frequência de implantações | maior |
| Lead Time for Changes | tempo entre commit e produção | menor |
| Change Failure Rate | percentual de mudanças que causam falha | menor |
| Time to Restore Service | tempo para restaurar o serviço | menor |


## 2. Modelo interno do Git

Git armazena objetos endereçados por hash:

- **blob:** conteúdo de um arquivo;
- **tree:** nomes, permissões e referências para blobs ou outras trees;
- **commit:** tree raiz, metadados, mensagem e commit(s) pai(s);
- **tag anotada:** nome de versão, mensagem e objeto apontado.

Uma branch é uma referência móvel para um commit. Uma referência como `origin/develop` registra a última visão local conhecida da branch remota após clone, fetch ou pull.


### Estados locais

```text
Working tree --git add--> Index --git commit--> Repositório local
```

- `git diff`: working tree comparada ao index;
- `git diff --staged`: index comparado ao último commit;
- `git status`: resumo da branch, arquivos e operação em andamento.


### Operações distribuídas

| Comando | Efeito |
|---|---|
| `git clone` | cria um repositório local, working tree e referências remotas |
| `git fetch` | baixa objetos e atualiza `origin/*`, sem alterar a working tree |
| `git pull` | executa fetch seguido de merge ou rebase |
| `git push` | envia objetos e solicita a atualização de uma referência remota |

Um push comum é rejeitado quando a atualização não é fast-forward. Isso evita apagar commits remotos que o cliente ainda não integrou.


## 3. Contratos do GitFlow

| Branch | Origem | Destino | Finalidade |
|---|---|---|---|
| `main` | permanente | — | estado de produção e tags |
| `develop` | permanente | — | integração da próxima versão |
| `feature/*` | `develop` | `develop` | mudança funcional isolada |
| `release/*` | `develop` | `main` e `develop` | estabilização e versionamento |
| `hotfix/*` | `main` | `main` e `develop` | correção urgente de produção |

O modelo adiciona governança, mas branches de longa duração aumentam divergência. Para alta frequência de entrega, Trunk-Based Development reduz esse custo com branches curtas e feature flags.


## 4. CI/CD, testes e gates

**Integração Contínua:** integrações frequentes verificadas por build e testes automáticos.

**Continuous Delivery:** o pipeline produz um artefato validado; a promoção final exige aprovação.

**Continuous Deployment:** o artefato aprovado pelos gates é promovido automaticamente.

Um pipeline contém eventos, workflows, jobs, runners e steps. Jobs independentes podem executar em paralelo; dependências explícitas definem a ordem.


### Pirâmide de testes e Quality Gate

| Tipo | Fronteira | Custo relativo |
|---|---|---|
| Unitário | função ou classe isolada | baixo |
| Integração | componentes reais controlados | médio |
| E2E | fluxo externo completo | alto |

Cobertura mede linhas ou ramos executados, não a qualidade das asserções. Um gate combina lint, testes, cobertura e políticas do repositório. O merge só fica bloqueado quando a branch protection exige os checks aprovados.


## 5. Laboratório distribuído com Git real

Topologia criada pelo notebook:

```text
clone_dev_a <--- push/fetch ---> origin.git (bare) <--- push/fetch ---> clone_dev_b
```

O diretório raiz é criado com `tempfile.mkdtemp`, fora do projeto. Nenhum repositório existente será alterado.


In [ ]:
from pathlib import Path
import subprocess
import tempfile
import shutil

def git(args, cwd=None, check=True):
    cmd = ["git", *args]
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    output = (result.stdout + result.stderr).strip()
    print(f"$ {' '.join(cmd)}")
    if output:
        print(output)
    if check and result.returncode != 0:
        raise RuntimeError(f"Comando falhou com código {result.returncode}")
    return result

if shutil.which("git") is None:
    raise EnvironmentError("Git não encontrado no PATH.")

print(git(["--version"]).stdout.strip())


### Etapa 1 — Criar o remoto bare

Um repositório bare não possui working tree. Servidores Git usam esse formato para receber e fornecer objetos e referências.


In [ ]:
lab_root = Path(tempfile.mkdtemp(prefix="gitflow_rede_"))
origin = lab_root / "origin.git"
dev_a = lab_root / "clone_dev_a"
dev_b = lab_root / "clone_dev_b"

git(["init", "--bare", "--initial-branch=main", str(origin)])
print("Laboratório:", lab_root)


### Etapa 2 — Primeiro clone e publicação de main/develop

O desenvolvedor A cria o primeiro commit, publica `main` e cria a branch de integração `develop`.


In [ ]:
git(["clone", str(origin), str(dev_a)])
git(["config", "user.name", "Dev A"], dev_a)
git(["config", "user.email", "dev.a@example.local"], dev_a)

(dev_a / "README.md").write_text("# Serviço de tarefas\n", encoding="utf-8")
git(["add", "README.md"], dev_a)
git(["commit", "-m", "docs: inicia serviço de tarefas"], dev_a)
git(["push", "-u", "origin", "main"], dev_a)
git(["switch", "-c", "develop"], dev_a)
git(["push", "-u", "origin", "develop"], dev_a)


### Etapa 3 — Segundo clone e inspeção das referências

O clone B possui seus próprios objetos, branches e working tree. O remoto apenas conecta os históricos.


In [ ]:
git(["clone", str(origin), str(dev_b)])
git(["config", "user.name", "Dev B"], dev_b)
git(["config", "user.email", "dev.b@example.local"], dev_b)
git(["fetch", "origin"], dev_b)
git(["switch", "develop"], dev_b)
git(["remote", "-v"], dev_b)
git(["branch", "-vv"], dev_b)
git(["ls-remote", "origin"], dev_b)


### Etapa 4 — Feature publicada pelo desenvolvedor A

A branch nasce de `develop`, recebe um commit atômico e é publicada com upstream.


In [ ]:
git(["switch", "develop"], dev_a)
git(["pull", "--ff-only"], dev_a)
git(["switch", "-c", "feature/filtro-status"], dev_a)
(dev_a / "filtro.txt").write_text("status=aberta|concluida\n", encoding="utf-8")
git(["add", "filtro.txt"], dev_a)
git(["commit", "-m", "feat(filtro): adiciona filtro por status"], dev_a)
git(["push", "-u", "origin", "feature/filtro-status"], dev_a)

git(["fetch", "origin"], dev_b)
git(["log", "--graph", "--oneline", "--decorate", "--all", "-n", "12"], dev_b)


### Etapa 5 — Integração da feature em develop

Neste laboratório, o merge ocorre por linha de comando. Em um serviço como GitHub, a operação normalmente ocorre após Pull Request, revisão e Quality Gates.


In [ ]:
git(["switch", "develop"], dev_a)
git(["merge", "--no-ff", "feature/filtro-status", "-m", "merge: integra filtro por status"], dev_a)
git(["push", "origin", "develop"], dev_a)

git(["pull", "--ff-only", "origin", "develop"], dev_b)
git(["log", "--graph", "--oneline", "--decorate", "--all", "-n", "12"], dev_b)


### Etapa 6 — Criar divergência entre os clones

Os dois desenvolvedores alteram o mesmo arquivo a partir da mesma base. O desenvolvedor B publica primeiro.


In [ ]:
git(["switch", "develop"], dev_a)
git(["pull", "--ff-only"], dev_a)
git(["switch", "develop"], dev_b)
git(["pull", "--ff-only"], dev_b)

(dev_a / "config.ini").write_text("timeout=30\n", encoding="utf-8")
git(["add", "config.ini"], dev_a)
git(["commit", "-m", "feat(config): define timeout de 30 segundos"], dev_a)

(dev_b / "config.ini").write_text("timeout=60\n", encoding="utf-8")
git(["add", "config.ini"], dev_b)
git(["commit", "-m", "feat(config): define timeout de 60 segundos"], dev_b)
git(["push", "origin", "develop"], dev_b)


### Etapa 7 — Observar a rejeição non-fast-forward

O remoto rejeita o push de A porque `origin/develop` contém um commit que A ainda não integrou.


In [ ]:
push_rejeitado = git(["push", "origin", "develop"], dev_a, check=False)
assert push_rejeitado.returncode != 0, "O push deveria ser rejeitado."
print("Rejeição confirmada: o histórico remoto não é ancestral do histórico local.")


### Etapa 8 — Fetch, merge e conflito

O fetch atualiza a visão remota. O merge tenta combinar os dois commits e pausa porque ambos adicionaram `config.ini` com conteúdos diferentes.


In [ ]:
git(["fetch", "origin"], dev_a)
git(["log", "--graph", "--oneline", "--decorate", "--all", "-n", "14"], dev_a)
merge_conflito = git(["merge", "origin/develop"], dev_a, check=False)
assert merge_conflito.returncode != 0, "O merge deveria produzir conflito."
git(["status", "--short"], dev_a)
print((dev_a / "config.ini").read_text(encoding="utf-8"))


### Etapa 9 — Resolver e publicar

A resolução é uma decisão de engenharia. O exemplo adota 45 segundos e registra explicitamente o merge.


In [ ]:
(dev_a / "config.ini").write_text("timeout=45\n", encoding="utf-8")
git(["add", "config.ini"], dev_a)
git(["commit", "-m", "merge: resolve timeout concorrente em 45 segundos"], dev_a)
git(["push", "origin", "develop"], dev_a)

git(["pull", "--ff-only", "origin", "develop"], dev_b)
git(["log", "--graph", "--oneline", "--decorate", "--all", "-n", "18"], dev_b)


### Etapa 10 — Release e tag SemVer

A release nasce de `develop`, integra `main` e retorna para `develop`. A tag anotada identifica o commit publicado.


In [ ]:
git(["switch", "develop"], dev_a)
git(["switch", "-c", "release/1.0.0"], dev_a)
(dev_a / "VERSION").write_text("1.0.0\n", encoding="utf-8")
git(["add", "VERSION"], dev_a)
git(["commit", "-m", "chore(release): prepara versão 1.0.0"], dev_a)

git(["switch", "main"], dev_a)
git(["merge", "--no-ff", "release/1.0.0", "-m", "release: publica versão 1.0.0"], dev_a)
git(["tag", "-a", "v1.0.0", "-m", "Versão 1.0.0"], dev_a)
git(["push", "origin", "main", "--follow-tags"], dev_a)

git(["switch", "develop"], dev_a)
git(["merge", "--no-ff", "release/1.0.0", "-m", "merge: retorna release 1.0.0 para develop"], dev_a)
git(["push", "origin", "develop"], dev_a)


### Auditoria final

Verifique referências remotas, tags e o grafo completo.


In [ ]:
git(["ls-remote", "--heads", "--tags", "origin"], dev_a)
git(["log", "--graph", "--oneline", "--decorate", "--all", "-n", "25"], dev_a)
print("Diretório do laboratório:", lab_root)


## 6. Questões técnicas

1. Por que `fetch` não altera a working tree?
2. Qual condição torna um push fast-forward?
3. Por que o conflito só apareceu durante a integração?
4. Qual informação de `git branch -vv` ajuda a diagnosticar sincronização?
5. Por que `release/*` deve retornar para `develop`?
6. Qual controle transforma o resultado do CI em bloqueio real de merge?
7. Qual diferença existe entre cobertura e qualidade das asserções?
8. Em qual cenário Trunk-Based Development reduz custos em relação ao GitFlow?


## 7. Gabarito resumido

1. Porque fetch atualiza objetos e referências remotas locais, sem executar checkout ou merge.
2. O commit remoto atual precisa ser ancestral do novo commit enviado.
3. Cada commit local era válido isoladamente; o conflito surgiu ao combinar alterações concorrentes na mesma região.
4. O upstream e os indicadores ahead/behind.
5. Para manter na próxima linha de desenvolvimento os ajustes realizados durante a estabilização.
6. Branch protection exigindo status checks aprovados.
7. Cobertura mede execução; asserções verificam resultados e invariantes.
8. Em produtos com integração diária, entrega frequente e uso de feature flags.


## Referências internas

- `Aulas/semana_01.ipynb` a `Aulas/semana_07.ipynb`
- `Materiais/guia_passo_a_passo_gitflow.md`
- `Materiais/padrao_de_commits.md`
- `Materiais/versionamento_semantico.md`
- `Materiais/todo-app/`
